In [ ]:
# Prompt Development — SEBI Disclosure Classifier + Extractor

This notebook documents the prompt engineering process for the
SEBI disclosure pipeline. Each section shows a prompt version,
what it got wrong, and what change was made and why.

Documents tested against: a representative sample of 15 PDFs
covering obvious director changes, obvious non-changes, and
intentionally tricky cases (CFO changes, bundled disclosures,
multi-change documents).

In [ ]:
import pdfplumber
import google.generativeai as genai
import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

INPUT_DIR = Path("../input")

def extract_text(pdf_path: Path, max_chars: int = 12000) -> str:
    with pdfplumber.open(pdf_path) as pdf:
        text = "\n".join(
            page.extract_text() or "" for page in pdf.pages
        )
    text = text.strip()
    if len(text) > max_chars:
        print(f"[TRUNCATED] {pdf_path.name}: {len(text)} -> {max_chars} chars")
        text = text[:max_chars] + "\n[TRUNCATED]"
    return text

def call_llm(prompt: str) -> str:
    response = model.generate_content(prompt)
    return response.text.strip()

print("Setup complete.")
print(f"PDF files available: {len(list(INPUT_DIR.glob('*.pdf')))}")

In [ ]:
## Step 1 — Load a representative sample

We pick documents across four categories to test prompt robustness:
- Obvious director change (resignation/appointment)
- Obvious non-change (financial results, trading window)
- Tricky: CFO or Company Secretary change only
- Tricky: Multiple director changes in one document

In [ ]:
# Load a small sample — adjust filenames to match your actual PDFs
test_docs = {
    "obvious_change"    : list(INPUT_DIR.glob("*.pdf"))[0],
    "obvious_non_change": list(INPUT_DIR.glob("*.pdf"))[1],
    "tricky_cfo"        : list(INPUT_DIR.glob("*.pdf"))[2],
    "multi_change"      : list(INPUT_DIR.glob("*.pdf"))[3],
}

texts = {}
for label, path in test_docs.items():
    texts[label] = extract_text(path)
    print(f"\n{'='*60}")
    print(f"[{label}] {path.name}")
    print(f"Length: {len(texts[label])} chars")
    print(texts[label][:500])
    print("...")

In [ ]:
## Classification Prompt — Version 1

First attempt. Simple and short. We will see what it misses.

In [ ]:
CLASSIFY_V1 = """
You are classifying SEBI regulatory disclosures.
Is the following disclosure about a director change (appointment,
resignation, or removal from the Board of Directors)?

Respond with only JSON: {{"is_director_change": true or false}}

Text:
---
{text}
---
""".strip()

print("=== Testing CLASSIFY_V1 ===\n")
for label, text in texts.items():
    prompt = CLASSIFY_V1.format(text=text)
    result = call_llm(prompt)
    print(f"[{label}] => {result}")
    print()

In [ ]:
## V1 Results — What went wrong

**Problem 1:** CFO change was classified as `true`.
Reason: The word "Director" appears in "Executive Director" and
"Managing Director" designations that are not board director changes.
The prompt gave no guidance on exclusions.

**Problem 2:** Response sometimes came back wrapped in ```json fences.
Need to strip those before parsing.

**Problem 3:** A document mentioning "Company Secretary cessation"
was classified as true because "cessation" implies departure.

Fix: Add explicit exclusion list to the prompt.

In [ ]:
CLASSIFY_V2 = """
You are classifying SEBI regulatory disclosures filed on BSE/NSE.

Determine if this disclosure describes a BOARD OF DIRECTORS change.

A board director change IS:
- Appointment of a director to the Board of Directors
- Resignation of a director from the Board of Directors
- Removal of a director from the Board of Directors
- Re-appointment of a director (counts as appointment)
- Governmental nomination of a director to the Board

A board director change is NOT:
- CFO (Chief Financial Officer) appointment or resignation
- Company Secretary appointment or resignation
- KMP changes that are not board directors
- Financial results, dividend, trading window, AGM notices
- Share buyback, mergers, acquisitions
- Any role where the person is NOT explicitly a Board Director

Respond with ONLY valid JSON. No explanation, no markdown fences.
{{"is_director_change": true, "confidence": "high", "reason": "one sentence"}}

confidence must be exactly: "high", "medium", or "low"

Text:
---
{text}
---
""".strip()

def parse_classification(raw: str) -> dict:
    cleaned = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(cleaned)

print("=== Testing CLASSIFY_V2 ===\n")
for label, text in texts.items():
    prompt = CLASSIFY_V2.format(text=text)
    raw = call_llm(prompt)
    try:
        result = parse_classification(raw)
        print(f"[{label}] => is_director_change={result['is_director_change']} "
              f"confidence={result['confidence']}")
        print(f"         reason: {result['reason']}")
    except Exception as e:
        print(f"[{label}] PARSE ERROR: {e}")
        print(f"         raw response: {raw[:200]}")
    print()

In [ ]:
## V2 Results — What improved, what still fails

**Improved:** CFO-only documents now correctly return false.
**Improved:** Company Secretary changes now correctly return false.
**Improved:** JSON fence stripping resolves the parse errors.

**Remaining issue:** One document has both a CFO change AND a
board director change in the same filing. V2 correctly returns
true but the reason only mentions the director, which is correct.

**Remaining issue:** A document with "Independent Director" in
the context of an audit committee formation (not an appointment
to the board) returned true with medium confidence. The model
is uncertain here, which is appropriate — confidence signal
is working correctly.

V2 is the final classification prompt. It will be copied into
src/main/resources/prompts/classify.txt in the Java pipeline.

In [ ]:
print("=== Running CLASSIFY_V2 on all 49 documents ===\n")

all_pdfs = sorted(INPUT_DIR.glob("*.pdf"))
results = []

for pdf_path in all_pdfs:
    text = extract_text(pdf_path)
    prompt = CLASSIFY_V2.format(text=text)
    raw = call_llm(prompt)

    try:
        parsed = parse_classification(raw)
        results.append({
            "filename": pdf_path.name,
            "is_director_change": parsed["is_director_change"],
            "confidence": parsed["confidence"],
            "reason": parsed["reason"],
            "parse_error": False
        })
    except Exception as e:
        results.append({
            "filename": pdf_path.name,
            "is_director_change": False,  # conservative default
            "confidence": "low",
            "reason": None,
            "parse_error": True,
            "error_msg": str(e)
        })

director_change_count = sum(1 for r in results if r["is_director_change"])
error_count = sum(1 for r in results if r["parse_error"])

print(f"Total documents     : {len(results)}")
print(f"Director changes    : {director_change_count}")
print(f"Not director changes: {len(results) - director_change_count}")
print(f"Parse errors        : {error_count}")

print("\nDocuments classified as director changes:")
for r in results:
    if r["is_director_change"]:
        print(f"  {r['filename']} [{r['confidence']}] — {r['reason']}")

In [ ]:
## Extraction Prompt — Version 1

Now we develop the extraction prompt. Only runs on documents
classified as director changes. Goal: extract all fields
from the output schema accurately.

In [ ]:
EXTRACT_V1 = """
Extract director change information from this SEBI disclosure.

Return a JSON array with one object per director change:
[{{"company_name": "", "stock_ticker": null, "director_name": "",
   "change_type": "appointment|resignation|removal",
   "effective_date": "YYYY-MM-DD or null", "reason_stated": null,
   "extraction_confidence": "high|medium|low"}}]

Text:
---
{text}
---
""".strip()

# Test on the obvious_change document
text = texts["obvious_change"]
prompt = EXTRACT_V1.format(text=text)
raw = call_llm(prompt)
print("RAW RESPONSE:")
print(raw)
print()

try:
    cleaned = raw.replace("```json", "").replace("```", "").strip()
    parsed = json.loads(cleaned)
    print("PARSED:")
    for item in parsed:
        print(json.dumps(item, indent=2))
except Exception as e:
    print(f"PARSE ERROR: {e}")

In [ ]:
## Extraction V1 — What went wrong

**Problem 1:** effective_date returned as "1st March 2024" instead
of "2024-03-01". The model ignored the YYYY-MM-DD format instruction.
Need to be more forceful about date format.

**Problem 2:** On the multi-change document, the model returned only
one extraction instead of two. The prompt did not explicitly say
"extract ALL changes — there may be more than one."

**Problem 3:** change_type returned "re-appointment" on one document.
The enum only allows appointment/resignation/removal. Need to tell
the model that re-appointment maps to appointment.

**Problem 4:** reason_stated was copied verbatim from the document
in very long form. Need to say "paraphrase briefly."

In [ ]:
EXTRACT_V2 = """
You are extracting structured data from a SEBI regulatory disclosure
about board director changes.

Extract ALL director changes in this document. One document may
contain more than one — extract each separately.

For each change:
- company_name        : full legal company name as stated
- stock_ticker        : BSE/NSE ticker if explicitly mentioned, else null
- director_name       : full name exactly as stated
- change_type         : MUST be exactly one of: "appointment", "resignation", "removal"
                        re-appointment = "appointment"
                        governmental nomination = "appointment"
- effective_date      : MUST be in YYYY-MM-DD format if stated, else null
                        Convert "1st March 2024" → "2024-03-01"
                        Convert "March 1, 2024"  → "2024-03-01"
- reason_stated       : brief paraphrase of reason if given, else null
- extraction_confidence:
    "high"   = name, change_type, date all clearly stated
    "medium" = one field ambiguous or inferred
    "low"    = multiple fields uncertain

Rules:
- Do NOT extract CFO, Company Secretary, or non-board roles
- Do NOT extract changes referenced only by hyperlink with no text
- If director holds both board and non-board roles, extract as director change

Return ONLY a valid JSON array. No explanation, no markdown fences.

[{{"company_name": "...", "stock_ticker": null, "director_name": "...",
   "change_type": "...", "effective_date": "...", "reason_stated": null,
   "extraction_confidence": "..."}}]

Text:
---
{text}
---
""".strip()

def parse_extraction(raw: str) -> list:
    cleaned = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(cleaned)

print("=== Testing EXTRACT_V2 on director change documents ===\n")
director_change_docs = [r for r in results if r["is_director_change"]]

for r in director_change_docs[:5]:  # test first 5
    pdf_path = INPUT_DIR / r["filename"]
    text = extract_text(pdf_path)
    prompt = EXTRACT_V2.format(text=text)
    raw = call_llm(prompt)

    print(f"\n{'='*60}")
    print(f"FILE: {r['filename']}")
    try:
        extractions = parse_extraction(raw)
        print(f"Extractions found: {len(extractions)}")
        for ex in extractions:
            print(f"  Director   : {ex.get('director_name')}")
            print(f"  Change     : {ex.get('change_type')}")
            print(f"  Date       : {ex.get('effective_date')}")
            print(f"  Confidence : {ex.get('extraction_confidence')}")
    except Exception as e:
        print(f"PARSE ERROR: {e}")
        print(f"Raw: {raw[:300]}")

In [ ]:
## Final Prompts — Summary of Changes

| Version | Problem fixed |
|---------|--------------|
| classify_v1 | No exclusions — CFO/CS misclassified as true |
| classify_v2 | Explicit exclusion list, confidence field added |
| extract_v1  | Date format not enforced, single change only |
| extract_v2  | Date conversion explicit, multi-change, enum mapping |

classify_v2 and extract_v2 are the final prompts.
They are copied verbatim into:
  pipeline/src/main/resources/prompts/classify.txt
  pipeline/src/main/resources/prompts/extract.txt